# Ten EPANET-MCP example prompts and results

This notebook extends the pattern in `02-Langchain-Agent.ipynb`: one persistent MCP session holds loaded networks, and tool calls operate inside that session. The main run uses direct client orchestration so each prompt has a reproducible answer and an ordered record of the tool calls. The final section shows how to run the same prompts through a LangChain agent and collect its answer and tool calls.

Run from this repository's `.venv` after `uv sync --extra examples`. `Net1.inp` is bundled with ePyT. The chosen IDs exist in that model: junction `11`, pipe `10`, and reservoir `9`. The target pressure of `107.5` is an illustrative threshold in the model's pressure units. Candidate diameters of 150–300 are exactly the values requested for the example and should be assessed in the model's units before engineering use.

Each numbered example starts from a freshly loaded `Net1.inp` under `baseline`. Scenario tools clone it; the notebook unloads each network after its answer is assembled. This keeps changes in the demand, diameter, and quality examples from carrying into later questions.


## Questions

The `questions` list holds the ten prompts. The placeholders have been replaced with IDs from Net1. Example 8 uses reservoir `9` because a concentration source at demand junction `11` does not introduce new water into this model.


In [1]:
questions = [
    "Load `Net1.inp` as `baseline`. Give me a summary of its nodes, links, flow units, simulation duration, and hydraulic time step.",
    "For `baseline`, list the junctions, reservoirs, tanks, pipes, pumps, and valves with their available properties. Also show the network's demand patterns and controls.",
    "In `baseline`, set the base demand of junction `11` to 15 in the model's flow units and change the diameter of pipe `10` to 200 in the model's diameter units. Show me the updated values.",
    "Run a hydraulic simulation of `baseline`. Report the minimum junction pressure over the simulation and identify the junction and time at which it occurs.",
    "Configure `baseline` for water-age analysis and run a water-quality simulation. Which junction has the highest water age at the final reporting time?",
    "Create a scenario from `baseline` in which the base demand at junction `11` is increased by 20%. Run the simulation and compare its minimum junction pressure with the baseline result. Leave the baseline unchanged.",
    "Create a leakage scenario on pipe `10`, using a target leak flow equal to 10% of its baseline mean absolute flow. Run the simulation and identify the junctions with the largest pressure decreases compared with the baseline.",
    "Create a contamination scenario with a concentration source of 10 mg/L at node `9`, starting at hour 2 and ending at hour 4. Check that the simulation duration and pattern timing support this interval, then run the scenario and report the nodes that receive nonzero contaminant concentrations and when they are first affected.",
    "Starting from `baseline` each time, create and simulate separate scenarios with demand multipliers of 1.0, 1.1, 1.2, and 1.3 at junction `11`. For each scenario, report the minimum junction pressure and where and when it occurs.",
    "Starting from `baseline`, test pipe `10` diameters of 150, 200, 250, and 300 in the model's diameter units. For each diameter, run a hydraulic simulation and report the minimum junction pressure over time. Identify the smallest tested diameter that keeps all junction pressures above 107.5 in the model's pressure units.",
]
assert len(questions) == 10


## MCP connection and result helpers

The server starts with the notebook's Python interpreter, so the installed repository version is used. `decode_mcp_result` turns LangChain MCP text blocks into Python dictionaries. Each recorded call contains its tool name and arguments in execution order; repetitions matter for the final two examples.


In [2]:
import json
import os

import pandas as pd
from IPython.display import display
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

client = MultiServerMCPClient({
    "epanet": {
        "transport": "stdio",
        "command": "uvx",
        "args": ["epanet-mcp-server"],
    }
})


def decode_mcp_result(result):
    if isinstance(result, list):
        parts = [item.get("text", "") for item in result if isinstance(item, dict) and item.get("type") == "text"]
        if len(parts) == 1:
            return json.loads(parts[0])
        return [json.loads(part) for part in parts]
    if isinstance(result, str):
        return json.loads(result)
    return result


def clock_time(seconds):
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}"


def junction_ids(nodes):
    return {node["id"] for node in nodes["nodes"] if str(node["type"]).upper() == "JUNCTION"}


def minimum_junction_pressure(simulation, junctions):
    return min(
        (float(value), simulation["node_ids"][column], int(simulation["time_steps_s"][row_index]))
        for row_index, row in enumerate(simulation["pressure"])
        for column, value in enumerate(row)
        if simulation["node_ids"][column] in junctions
    )


def minimum_text(minimum):
    pressure, node_id, seconds = minimum
    return f"{pressure:.3f} pressure units at junction {node_id} at {clock_time(seconds)} ({seconds} s)"


## Run the ten examples

The code below answers the questions from MCP results. For the contamination example, Net1 already has nonzero initial chemical quality. “First affected” therefore means the first reporting time with a positive concentration *increase over the unmodified baseline* (tolerance `1e-3` mg/L). Without that comparison, all nodes would be incorrectly reported as affected at time zero. The leakage comparison reports the largest instantaneous pressure drop at each junction over matching reporting times.


In [6]:
from tqdm import tqdm

async def run_agent_examples():
    from dotenv import load_dotenv
    from langchain.agents import create_agent
    from langchain_openai import ChatOpenAI

    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY")
    base_url = os.getenv("OPENAI_BASE_URL")
    if not api_key or not base_url:
        raise RuntimeError("Set OPENAI_API_KEY and OPENAI_BASE_URL before running the agent examples.")

    model = ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-5.6-luna"),
        api_key=api_key,
        base_url=base_url,
    )
    rows = []
    async with client.session("epanet") as session:
        available = {tool.name: tool for tool in await load_mcp_tools(session)}
        agent = create_agent(
            model=model,
            tools=list(available.values()),
            system_prompt=(
                "Use EPANET-MCP tools for all model facts and calculations. Exclude tanks and reservoirs "
                "from junction-pressure extrema. For scenario comparisons use matching reporting times. "
                "Net1 has nonzero initial chemical quality; for contamination, compare against a baseline "
                "quality run before deciding when new contamination arrives. For questions 9 and 10, "
                "make a separate series of tool calls for every candidate and report all candidates."
            ),
        )
        for number, question in enumerate(tqdm(questions), start=1):
            setup_calls = []
            if number > 1:
                arguments = {"path": "Net1.inp", "network_id": "baseline"}
                decode_mcp_result(await available["load_network"].ainvoke(arguments))
                setup_calls.append({"name": "load_network", "args": arguments, "origin": "notebook setup"})
            try:
                result = await agent.ainvoke(
                    {"messages": [{"role": "user", "content": question}]},
                    config={"recursion_limit": 100},
                )
                tool_calls = setup_calls + [
                    {"name": call["name"], "args": call["args"]}
                    for message in result["messages"]
                    for call in getattr(message, "tool_calls", [])
                ]
                rows.append({
                    "question": question,
                    "final answer": result["messages"][-1].content,
                    "set of tool calls": tool_calls,
                })
            finally:
                loaded = decode_mcp_result(await available["list_networks"].ainvoke({}))["loaded_networks"]
                for network_id in loaded:
                    await available["unload_network"].ainvoke({"network_id": network_id})
    return pd.DataFrame(rows, columns=["question", "final answer", "set of tool calls"])

df = await run_agent_examples()

100%|██████████| 10/10 [04:03<00:00, 24.32s/it]


In [28]:
def get_calls(calls):
    calls = [x['name'] for x in calls]
    return list(map(lambda x: f'\\texttt' + '{' + x + '}', set(calls)))

df['calls'] = df['set of tool calls'].apply(get_calls)
print(df[['question', 'final answer', 'calls']].to_latex(index=False))

\begin{tabular}{lll}
\toprule
question & final answer & calls \\
\midrule
Load `Net1.inp` as `baseline`. Give me a summary of its nodes, links, flow units, simulation duration, and hydraulic time step. & `Net1.inp` is loaded as **`baseline`**.

- **Nodes:** 11 total  
  - 9 junctions  
  - 1 tank  
  - 1 reservoir
- **Links:** 13 total  
  - 12 pipes  
  - 1 pump  
  - 0 valves
- **Flow units:** GPM
- **Simulation duration:** 86,400 seconds (24 hours)
- **Hydraulic time step:** 3,600 seconds (1 hour) & ['\\texttt{load_network}'] \\
For `baseline`, list the junctions, reservoirs, tanks, pipes, pumps, and valves with their available properties. Also show the network's demand patterns and controls. & ## `baseline` network inventory

### Junctions

| ID | Elevation | Base demand |
|---|---:|---:|
| 10 | 710.0 | 0.0 |
| 11 | 710.0 | 150.0 |
| 12 | 700.0 | 150.0 |
| 13 | 695.0 | 100.0 |
| 21 | 700.0 | 150.0 |
| 22 | 695.0 | 200.0 |
| 23 | 690.0 | 150.0 |
| 31 | 700.0 | 100.0 |
| 32 | 710.0 |